# Use `fct_monthly_routes` to make legislative district crosswalk

In [1]:
import geopandas as gpd
import gcsfs
import google.auth
import pandas as pd

from update_vars import DIGEST_DICT, RAW_GCS, abbrev_month

credentials, _ = google.auth.default()

In [2]:
filename = f"{RAW_GCS}{DIGEST_DICT.route_map}_{abbrev_month}.parquet"

In [3]:
gdf = gpd.read_parquet(
    filename, storage_options = {"token": credentials.token},
    columns = [
        "schedule_name", "analysis_name", "geometry"
    ]
)

# fct_monthly_routes for digest contains 2 months, current month and prior month

In [4]:
gdf.head(2)

,schedule_name,analysis_name,geometry
0,Bay Area 511 SamTrans Schedule,San Mateo County Transit District,"LINESTRING (-122.42774 37.62222, -122.42771 37..."
1,GET Schedule,Golden Empire Transit District,"LINESTRING (-118.97323 35.41105, -118.97251 35..."


In [5]:
SHARED_GCS = "gs://calitp-analytics-data/data-analyses/shared_data/"
legislative_districts = gpd.read_parquet(
    f"{SHARED_GCS}legislative_districts.parquet",
    storage_options = {"token": credentials.token}
)

In [6]:
def sjoin_shapes_legislative_districts(
    shapes_gdf: gpd.GeoDataFrame
) -> pd.DataFrame:
    """
    Grab shapes (fct_monthly_routes or daily shapes) and do a spatial join
    with legislative district.
    Keep 1 row for every operator-legislative_district combination.
    """
    legislative_districts = gpd.read_parquet(
        f"{SHARED_GCS}legislative_districts.parquet",
        storage_options = {"token": credentials.token}
    )
    
    crosswalk = gpd.sjoin(
        shapes_gdf, legislative_districts, 
        how="inner", 
        predicate="intersects"
    )[["schedule_name", "analysis_name", "legislative_district"]
    ].drop_duplicates().sort_values(
        ["schedule_name", "legislative_district"]
    ).reset_index(drop=True)

    return crosswalk

In [7]:
crosswalk = sjoin_shapes_legislative_districts(gdf)

In [8]:
crosswalk[["schedule_name", "analysis_name"]].drop_duplicates().shape

(152, 2)

In [ ]:
# all operators were mapped onto crosswalk, good, and most span multiple (expected)
gdf[["schedule_name", "analysis_name"]].drop_duplicates().shape

In [9]:
crosswalk.head(10)

,schedule_name,analysis_name,legislative_district
0,Alhambra Schedule,City of Alhambra,AD 49
1,Alhambra Schedule,City of Alhambra,AD 52
2,Alhambra Schedule,City of Alhambra,SD 25
3,Alhambra Schedule,City of Alhambra,SD 26
4,Amador Schedule,Amador Regional Transit System,AD 01
5,Amador Schedule,Amador Regional Transit System,AD 09
6,Amador Schedule,Amador Regional Transit System,SD 04
7,Antelope Valley Transit Authority Schedule,Antelope Valley Transit Authority,AD 34
8,Antelope Valley Transit Authority Schedule,Antelope Valley Transit Authority,AD 39
9,Antelope Valley Transit Authority Schedule,Antelope Valley Transit Authority,AD 40
